In [ ]:
!nvidia-smi

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

In [ ]:
%cd /content
!if [ ! -d /content/myRetinexformer/.git ]; then git clone https://github.com/Silly-Gooses/myRetinexformer.git /content/myRetinexformer; fi
import subprocess
for command in (
    ["git", "fetch", "origin", "v1"],
    ["git", "switch", "v1"],
    ["git", "pull", "--ff-only", "origin", "v1"],
):
    subprocess.run(command, cwd="/content/myRetinexformer", check=True)
%cd /content/myRetinexformer
from pathlib import Path
assert Path("analysis/compare_author_vs_retrained.py").is_file(), "Push the updated project to GitHub before running this notebook."
print("Version 1 branch selected; baseline analysis script also available")


In [ ]:
!pip install -q einops opencv-python pillow matplotlib


In [ ]:
from RetinexFormer_arch import RetinexFormer

print("RetinexFormer import successful")

# Model creation test

In [ ]:
import torch
from RetinexFormer_arch import RetinexFormer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = RetinexFormer(
    in_channels=3,
    out_channels=3,
    n_feat=40,
    stage=1,
    num_blocks=[1, 2, 2],
    edge_guidance=True
).to(device)

print("Device:", device)
print("Model created successfully")

params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {params:,}")

# Forward pass

In [ ]:
x = torch.rand(1, 3, 256, 256).to(device)

model.eval()

with torch.no_grad():
    y = model(x)

print("Input :", x.shape)
print("Output:", y.shape)
print("Forward pass successful")

In [ ]:
from dataLoader import paired_image_Dataset, get_dataloaders

print("DataLoader import successful")

In [ ]:
from analysis_utils import evaluate, export_test_set, infer_full_image, save_comparisons
from utils import PSNR, calculate_ssim

print("Project analysis imports successful")


# training backward pass

In [ ]:
import torch
import torch.nn as nn

model.train()

# Fake low-light image
x = torch.rand(1, 3, 256, 256).to(device)

# Fake ground-truth image
gt = torch.rand(1, 3, 256, 256).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=2e-4
)

criterion = nn.L1Loss()

optimizer.zero_grad()

output = model(x)
loss = criterion(output, gt)

loss.backward()
optimizer.step()

print("Output shape:", output.shape)
print("Loss:", loss.item())
print("Backward pass successful")
print("Optimizer step successful")

# Drive mount

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

dataset_root = Path(
    "/content/drive/MyDrive/datasets/LOLv2/Real_captured"
)

weight_path = Path(
    "/content/drive/MyDrive/RetinexFormer_weights/LOL_v2_real.pth"
)

train_low = dataset_root / "Train/Low"
train_normal = dataset_root / "Train/Normal"
test_low = dataset_root / "Test/Low"
test_normal = dataset_root / "Test/Normal"

print("=== Dataset ===")
print("Train Low    :", train_low.exists(), len(list(train_low.glob("*"))))
print("Train Normal :", train_normal.exists(), len(list(train_normal.glob("*"))))
print("Test Low     :", test_low.exists(), len(list(test_low.glob("*"))))
print("Test Normal  :", test_normal.exists(), len(list(test_normal.glob("*"))))

print("\n=== Weight ===")
print("Weight exists:", weight_path.exists())

if weight_path.exists():
    print(
        "Weight size:",
        round(weight_path.stat().st_size / 1024 / 1024, 2),
        "MB"
    )

In [ ]:
from dataLoader import paired_image_Dataset
from torch.utils.data import DataLoader

train_dataset = paired_image_Dataset(
    low_dir=str(train_low),
    normal_dir=str(train_normal),
    split='train',
    crop_size=(256, 256)
)

print("Dataset length:", len(train_dataset))

sample = train_dataset[0]

print("Keys:", sample.keys())
print("Low shape :", sample['low'].shape)
print("High shape:", sample['high'].shape)
print("Name      :", sample['name'])
print("Low range :", sample['low'].min().item(), sample['low'].max().item())
print("High range:", sample['high'].min().item(), sample['high'].max().item())

# Compare low-light image vs normal/GT image

In [ ]:
import matplotlib.pyplot as plt

sample = train_dataset[0]

low_img = sample['low'].permute(1, 2, 0).cpu().numpy()
high_img = sample['high'].permute(1, 2, 0).cpu().numpy()

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(low_img)
plt.title(f"Low - {sample['name']}")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(high_img)
plt.title(f"Normal - {sample['name']}")
plt.axis("off")

plt.show()

# 10-iteration smoke training

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

model.train()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=2e-4
)

criterion = nn.L1Loss()

max_iters = 10
current_iter = 0

for batch in train_loader:
    low = batch['low'].to(device)
    high = batch['high'].to(device)

    optimizer.zero_grad()

    output = model(low)
    loss = criterion(output, high)

    loss.backward()
    optimizer.step()

    current_iter += 1

    print(
        f"Iter {current_iter}/{max_iters} | "
        f"Loss: {loss.item():.6f}"
    )

    if current_iter >= max_iters:
        break

print("10-iteration smoke training successful")

# Checkpoint test

In [ ]:
import torch
from pathlib import Path

checkpoint_path = Path("/content/retinexformer_v1_smoke_checkpoint.pth")

torch.save({
    "model": model.state_dict(),
    "optimizer": optimizer.state_dict(),
    "iter": current_iter,
    "loss": loss.item(),
    "config": {
        "version": "v1", "edge_guidance": model.edge_guidance,
        "canny": model.canny_config,
        "architecture": {"in_channels": 3, "out_channels": 3, "n_feat": 40,
                         "stage": 1, "num_blocks": [1, 2, 2]},
    }
}, checkpoint_path)

print("Saved:", checkpoint_path)
print("Exists:", checkpoint_path.exists())
print("Size MB:", checkpoint_path.stat().st_size / 1024 / 1024)

In [ ]:
checkpoint = torch.load(
    "/content/retinexformer_v1_smoke_checkpoint.pth",
    map_location=device
)

model.load_state_dict(checkpoint["model"])
optimizer.load_state_dict(checkpoint["optimizer"])

print("Loaded iteration:", checkpoint["iter"])
print("Loaded loss:", checkpoint["loss"])
print("Checkpoint load successful")

# Validation test (average PSNR + SSIM)

In [ ]:
from dataLoader import paired_image_Dataset
from torch.utils.data import DataLoader

test_dataset = paired_image_Dataset(
    low_dir=str(test_low),
    normal_dir=str(test_normal),
    split="test",
    crop_size=None,
    augment=False,
)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2)
print("Test dataset length:", len(test_dataset))


In [ ]:
# Both metrics are calculated from the same 8-bit pixels used in PNG exports.
avg_psnr, avg_ssim = evaluate(model, test_loader, device)
print("\n=== Validation Result ===")
print(f"Average PSNR: {avg_psnr:.4f}")
print(f"Average SSIM: {avg_ssim:.4f}")


# test image output shape

In [ ]:
from PIL import Image

for i, path in enumerate(sorted(test_low.glob("*"))[:10]):
    original = Image.open(path)

    sample = test_dataset[i]
    _, h, w = sample['low'].shape

    status = "✅ SAME" if original.size == (w, h) else "⚠️ CHANGED"

    print(
        f"{path.name} | "
        f"original={original.size} | "
        f"loaded={(w, h)} | "
        f"{status}"
    )

# debug method

In [ ]:
import importlib
import RetinexFormer_arch

importlib.reload(RetinexFormer_arch)

from RetinexFormer_arch import RetinexFormer

In [ ]:
model = RetinexFormer(
    in_channels=3,
    out_channels=3,
    n_feat=40,
    stage=1,
    num_blocks=[1, 2, 2],
    edge_guidance=True
).to(device)

checkpoint = torch.load(
    "/content/retinexformer_v1_smoke_checkpoint.pth",
    map_location=device
)

model.load_state_dict(checkpoint["model"])

model.eval()

print("Smoke checkpoint loaded")

In [ ]:
from pathlib import Path
BASE_DIR = Path("/content/drive/MyDrive/RetinexFormer_v1_smoke_outputs")
print("Output folder:", BASE_DIR)


In [ ]:
NUM_IMAGES = 10
saved_dir = save_comparisons(
    model, test_dataset, range(min(NUM_IMAGES, len(test_dataset))), device, BASE_DIR
)
print("Saved comparisons to:", saved_dir)


In [ ]:
# Version 1: inspect structural guidance alongside RGB intermediates.
import matplotlib.pyplot as plt
import torch.nn.functional as F

sample = test_dataset[0]
low = sample["low"].unsqueeze(0).to(device)
height, width = low.shape[-2:]
pad_h, pad_w = (-height) % 4, (-width) % 4
pad_mode = "reflect" if pad_h < height and pad_w < width else "replicate"
padded = F.pad(low, (0, pad_w, 0, pad_h), mode=pad_mode)
with torch.inference_mode():
    details = model.forward_with_intermediate(padded, return_dict=True)
images = [(details[key][0, :, :height, :width], title) for key, title in (
    ("input", "Low-light RGB"), ("edge", "Canny edge"),
    ("lit_up", "Lit-up RGB"), ("output", "Enhanced RGB"),
)]
plt.figure(figsize=(18, 5))
for i, (img, title) in enumerate(images):
    plt.subplot(1, 4, i + 1)
    pixels = img.detach().float().clamp(0, 1).cpu()
    if pixels.shape[0] == 1:
        plt.imshow(pixels[0].numpy(), cmap="gray", vmin=0, vmax=1)
    else:
        plt.imshow(pixels.permute(1, 2, 0).numpy())
    plt.title(title)
    plt.axis("off")
plt.show()


# Version 1 Full Training — Google Colab (LOLv2 Real)

This section starts a **fresh model** for full training. It does not continue from the 10-iteration smoke-test model unless `RESUME_FROM` is set to a full-training checkpoint.

Default experiment settings:
- Total iterations: **150,000**
- Batch size: **8**
- Patch size: **256 × 256**
- Validation: every **1,000 iterations**
- Checkpoint: every **1,000 iterations**
- Fixed visual comparison: every **1,000 iterations**
- Loss: **L1 / MAE** (same loss used by this notebook's smoke test)

Canny guidance is enabled; v1 outputs use separate paths. Baseline optimizer checkpoints cannot be resumed into v1.


In [ ]:
# ============================================================
# FULL TRAINING CONFIG + FRESH MODEL + DATALOADERS
# ============================================================
import os
import csv
import math
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from RetinexFormer_arch import RetinexFormer
from dataLoader import paired_image_Dataset

# ---------- Reproducibility ----------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ---------- Experiment ----------
EXPERIMENT_NAME = "Lightweight_RetinexFormer_v1_LOLv2_real_Full"
TOTAL_ITERS = 150_000
BATCH_SIZE = 8
PATCH_SIZE = 256
BASE_LR = 2e-4
MIN_LR = 1e-6
VAL_FREQ = 1_000
CHECKPOINT_FREQ = 1_000
VIS_FREQ = 1_000
PRINT_FREQ = 100
NUM_WORKERS = 2
FIXED_VIS_COUNT = 5

# Keep False for the clean full-precision setup used by this notebook.
# Set True only if you intentionally want mixed precision for speed/memory.
USE_AMP = False

# ---------- Google Drive paths ----------
# These reuse the dataset paths already checked earlier in the notebook.
FULL_ROOT = Path("/content/drive/MyDrive/RetinexFormer_full_training") / EXPERIMENT_NAME
CKPT_DIR = FULL_ROOT / "checkpoints"
VIS_DIR = FULL_ROOT / "visuals"
LOG_DIR = FULL_ROOT / "logs"
for d in (CKPT_DIR, VIS_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

METRICS_CSV = LOG_DIR / "training_metrics.csv"

# To resume, put a checkpoint path here, e.g.
# RESUME_FROM = CKPT_DIR / "iter_050000.pth"
RESUME_FROM = None

# ---------- Dataset ----------
# train_low/train_normal/test_low/test_normal are defined by the earlier
# Google Drive dataset-path cell in this notebook.
train_dataset_full = paired_image_Dataset(
    low_dir=str(train_low),
    normal_dir=str(train_normal),
    split="train",
    crop_size=(PATCH_SIZE, PATCH_SIZE)
)

test_dataset_full = paired_image_Dataset(
    low_dir=str(test_low),
    normal_dir=str(test_normal),
    split="test",
    crop_size=None,
    augment=False
)

train_loader_full = DataLoader(
    train_dataset_full,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
    persistent_workers=(NUM_WORKERS > 0)
)

test_loader_full = DataLoader(
    test_dataset_full,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0)
)

print("Train images:", len(train_dataset_full))
print("Test images :", len(test_dataset_full))
print("Batches/epoch:", len(train_loader_full))
print("Output root:", FULL_ROOT)

# ---------- Fresh model ----------
# IMPORTANT: this intentionally discards the smoke-test model state.
model_full = RetinexFormer(
    in_channels=3,
    out_channels=3,
    n_feat=40,
    stage=1,
    num_blocks=[1, 2, 2],
    edge_guidance=True
).to(device)

criterion_full = nn.L1Loss()
optimizer_full = torch.optim.Adam(
    model_full.parameters(),
    lr=BASE_LR,
    betas=(0.9, 0.999)
)

# One cosine decay over the full 150k iterations.
scheduler_full = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_full,
    T_max=TOTAL_ITERS,
    eta_min=MIN_LR
)

# AMP API compatible with current Colab PyTorch.
scaler_full = torch.amp.GradScaler("cuda", enabled=(USE_AMP and device.type == "cuda"))

params = sum(p.numel() for p in model_full.parameters())
print(f"Model parameters: {params:,}")
print("AMP enabled:", scaler_full.is_enabled())


In [ ]:
# ============================================================
# FULL TRAINING HELPERS: checkpoint and CSV logging
# ============================================================
from analysis_utils import evaluate, export_test_set, save_comparisons


def save_full_checkpoint(path, iteration, epoch, best_psnr, best_ssim):
    payload = {
        "model": model_full.state_dict(),
        "optimizer": optimizer_full.state_dict(),
        "scheduler": scheduler_full.state_dict(),
        "scaler": scaler_full.state_dict(),
        "iter": int(iteration),
        "epoch": int(epoch),
        "best_psnr": float(best_psnr),
        "best_ssim": float(best_ssim),
        "config": {
            "version": "v1",
            "edge_guidance": model_full.edge_guidance,
            "canny": model_full.canny_config,
            "architecture": {"in_channels": 3, "out_channels": 3, "n_feat": 40,
                             "stage": 1, "num_blocks": [1, 2, 2]},
            "total_iters": TOTAL_ITERS,
            "batch_size": BATCH_SIZE,
            "patch_size": PATCH_SIZE,
            "base_lr": BASE_LR,
            "min_lr": MIN_LR,
            "val_freq": VAL_FREQ,
            "checkpoint_freq": CHECKPOINT_FREQ,
            "use_amp": USE_AMP,
            "seed": SEED,
        },
    }
    torch.save(payload, path)


def append_metrics(iteration, epoch, train_loss, lr, val_psnr=None, val_ssim=None):
    new_file = not METRICS_CSV.exists()
    with open(METRICS_CSV, "a", newline="") as f:
        writer = csv.writer(f)
        if new_file:
            writer.writerow(["iter", "epoch", "train_loss", "lr", "val_psnr", "val_ssim"])
        writer.writerow([
            int(iteration), int(epoch), float(train_loss), float(lr),
            "" if val_psnr is None else float(val_psnr),
            "" if val_ssim is None else float(val_ssim),
        ])


# Use the same test images at every visualization checkpoint.
if len(test_dataset_full) <= FIXED_VIS_COUNT:
    FIXED_VIS_INDICES = list(range(len(test_dataset_full)))
else:
    FIXED_VIS_INDICES = np.linspace(
        0, len(test_dataset_full) - 1, FIXED_VIS_COUNT, dtype=int
    ).tolist()
print("Fixed visual indices:", FIXED_VIS_INDICES)


In [ ]:
# ============================================================
# OPTIONAL RESUME
# ============================================================
start_iter = 0
start_epoch = 0
best_psnr = -float("inf")
best_ssim = -float("inf")

if RESUME_FROM is not None:
    RESUME_FROM = Path(RESUME_FROM)
    if not RESUME_FROM.exists():
        raise FileNotFoundError(f"Resume checkpoint not found: {RESUME_FROM}")

    ckpt = torch.load(RESUME_FROM, map_location=device)
    if (ckpt.get("config", {}).get("version") != "v1"
            or ckpt["config"].get("canny") != model_full.canny_config):
        raise ValueError("Resume requires a v1 checkpoint with matching Canny settings")
    model_full.load_state_dict(ckpt["model"])
    optimizer_full.load_state_dict(ckpt["optimizer"])
    scheduler_full.load_state_dict(ckpt["scheduler"])

    if "scaler" in ckpt:
        scaler_full.load_state_dict(ckpt["scaler"])

    start_iter = int(ckpt.get("iter", 0))
    start_epoch = int(ckpt.get("epoch", 0))
    best_psnr = float(ckpt.get("best_psnr", -float("inf")))
    best_ssim = float(ckpt.get("best_ssim", -float("inf")))

    print("Resumed from:", RESUME_FROM)
    print("Start iter  :", start_iter)
    print("Start epoch :", start_epoch)
    print("Best PSNR   :", best_psnr)
    print("Best SSIM   :", best_ssim)
else:
    print("Starting full training from scratch.")


In [ ]:
# ============================================================
# RUN FULL TRAINING
# ============================================================
import time

current_iter = start_iter
epoch = start_epoch
running_loss = 0.0
running_count = 0
training_start = time.time()

model_full.train()

while current_iter < TOTAL_ITERS:
    epoch += 1

    for batch in train_loader_full:
        if current_iter >= TOTAL_ITERS:
            break

        low = batch["low"].to(device, non_blocking=True)
        high = batch["high"].to(device, non_blocking=True)

        optimizer_full.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type=device.type,
            enabled=(USE_AMP and device.type == "cuda")
        ):
            output = model_full(low)
            loss = criterion_full(output, high)

        if scaler_full.is_enabled():
            scaler_full.scale(loss).backward()
            scaler_full.step(optimizer_full)
            scaler_full.update()
        else:
            loss.backward()
            optimizer_full.step()

        scheduler_full.step()
        current_iter += 1

        loss_value = float(loss.item())
        running_loss += loss_value
        running_count += 1

        # Console/log every PRINT_FREQ
        if current_iter % PRINT_FREQ == 0:
            avg_train_loss = running_loss / max(running_count, 1)
            lr_now = optimizer_full.param_groups[0]["lr"]
            elapsed_h = (time.time() - training_start) / 3600.0

            print(
                f"Epoch {epoch:04d} | Iter {current_iter:06d}/{TOTAL_ITERS} | "
                f"Loss {avg_train_loss:.6f} | LR {lr_now:.8f} | "
                f"Elapsed {elapsed_h:.2f}h"
            )

            append_metrics(
                current_iter, epoch, avg_train_loss, lr_now,
                val_psnr=None, val_ssim=None
            )
            running_loss = 0.0
            running_count = 0

        # Validation every 1000 iterations
        if current_iter % VAL_FREQ == 0:
            val_psnr, val_ssim = evaluate(model_full, test_loader_full, device)
            lr_now = optimizer_full.param_groups[0]["lr"]

            print(
                f"[VAL] Iter {current_iter:06d} | "
                f"PSNR {val_psnr:.4f} | SSIM {val_ssim:.4f}"
            )

            append_metrics(
                current_iter, epoch, loss_value, lr_now,
                val_psnr=val_psnr, val_ssim=val_ssim
            )

            # Save best-by-PSNR model.
            if val_psnr > best_psnr:
                best_psnr = val_psnr
                best_ssim = val_ssim
                save_full_checkpoint(
                    CKPT_DIR / "best_psnr.pth",
                    current_iter, epoch, best_psnr, best_ssim
                )
                print(
                    f"[BEST] PSNR {best_psnr:.4f}, SSIM {best_ssim:.4f} "
                    f"at iter {current_iter}"
                )

        # Fixed 5-image Low/Input/Output/GT comparison every 1000 iters
        if current_iter % VIS_FREQ == 0:
            save_comparisons(
                model_full, test_dataset_full, FIXED_VIS_INDICES, device,
                VIS_DIR / f"iter_{current_iter:06d}"
            )
            print(f"[VIS] Saved fixed comparisons for iter {current_iter}")

        # Full resume-able checkpoint every 1000 iterations
        if current_iter % CHECKPOINT_FREQ == 0:
            ckpt_path = CKPT_DIR / f"iter_{current_iter:06d}.pth"
            save_full_checkpoint(
                ckpt_path, current_iter, epoch, best_psnr, best_ssim
            )
            print("[CKPT] Saved:", ckpt_path)

# Always save final checkpoint too.
final_path = CKPT_DIR / f"final_iter_{current_iter:06d}.pth"
save_full_checkpoint(final_path, current_iter, epoch, best_psnr, best_ssim)

print("\n=== FULL TRAINING COMPLETE ===")
print("Final iteration:", current_iter)
print("Final epoch    :", epoch)
print("Best PSNR      :", best_psnr)
print("Best SSIM      :", best_ssim)
print("Final checkpoint:", final_path)
print("Metrics CSV    :", METRICS_CSV)
print("Visuals        :", VIS_DIR)


## Final Test-Set Export (run after full training)
Loads the **best-PSNR checkpoint** and saves enhanced outputs for the entire LOLv2 Real test set, together with per-image PSNR/SSIM and an overall summary.


In [ ]:
# ============================================================
# FINAL TEST-SET EXPORT USING BEST CHECKPOINT
# ============================================================
FINAL_TEST_DIR = FULL_ROOT / "final_test_outputs"
best_ckpt_path = CKPT_DIR / "best_psnr.pth"
if not best_ckpt_path.exists():
    raise FileNotFoundError(f"Best checkpoint not found: {best_ckpt_path}")

best_ckpt = torch.load(best_ckpt_path, map_location=device)
model_full.load_state_dict(best_ckpt["model"])
avg_psnr, avg_ssim = export_test_set(
    model_full, test_loader_full, device, FINAL_TEST_DIR
)
print("=== FINAL TEST EXPORT COMPLETE ===")
print("Checkpoint :", best_ckpt_path)
print("Best iter  :", best_ckpt.get("iter"))
print(f"Avg PSNR   : {avg_psnr:.4f}")
print(f"Avg SSIM   : {avg_ssim:.4f}")
print("Outputs    :", FINAL_TEST_DIR / "output")
print("Metrics CSV:", FINAL_TEST_DIR / "per_image_metrics.csv")


## Author vs. Retrained Checkpoint Analysis

For **analysis only**, run cells 1, 2, and 13 first, then set the paths in the next cell. You do not need to run any model-training cells. Run the five-image smoke comparison before the full test set. Final outputs are scored against ground truth; Lit-up outputs use neutral diagnostic measurements.

This section deliberately compares **two baseline models** with edge guidance disabled. Keep its checkpoint paths pointing to baseline runs.


In [ ]:
# Set these Google Drive paths for your own experiment.
AUTHOR_CKPT = Path("/content/drive/MyDrive/RetinexFormer_weights/LOL_v2_real.pth")
# Use an already-trained checkpoint on Drive; this does not require any training cell.
RETRAINED_CKPT = Path(
    "/content/drive/MyDrive/RetinexFormer_full_training/"
    "Lightweight_RetinexFormer_LOLv2_real_Full/checkpoints/best_psnr.pth"
)
LOW_DIR = Path("/content/drive/MyDrive/datasets/LOLv2/Real_captured/Test/Low")
GT_DIR = Path("/content/drive/MyDrive/datasets/LOLv2/Real_captured/Test/Normal")
ANALYSIS_ROOT = Path("/content/drive/MyDrive/retinexformer_analysis")
for path in (AUTHOR_CKPT, RETRAINED_CKPT, LOW_DIR, GT_DIR):
    if not path.exists():
        raise FileNotFoundError(f"Update this analysis path: {path}")

!python analysis/compare_author_vs_retrained.py --author_ckpt {AUTHOR_CKPT} --retrained_ckpt {RETRAINED_CKPT} --low_dir {LOW_DIR} --gt_dir {GT_DIR} --output_dir {ANALYSIS_ROOT / 'smoke'} --max_images 5

print("Inspect:", ANALYSIS_ROOT / "smoke" / "per_image_analysis.csv")
print("Then run the next cell only after reviewing the smoke results.")


In [ ]:
# Full LOLv2 Real comparison (run after the smoke results look correct).
!python analysis/compare_author_vs_retrained.py --author_ckpt {AUTHOR_CKPT} --retrained_ckpt {RETRAINED_CKPT} --low_dir {LOW_DIR} --gt_dir {GT_DIR} --output_dir {ANALYSIS_ROOT / 'full'}
print("Full per-image analysis:", ANALYSIS_ROOT / "full" / "per_image_analysis.csv")


## Resume after Colab disconnect

1. Mount Drive and run the setup/import/dataset cells above.
2. In **FULL TRAINING CONFIG**, set `RESUME_FROM` to the last saved file, for example:
   `CKPT_DIR / "iter_050000.pth"`
3. Run the full-training helper/resume/training cells again.

Do **not** resume from `retinexformer_v1_smoke_checkpoint.pth`; that checkpoint belongs only to the 10-iteration smoke test.


## Optional v1 correctness and timing checks
Run after setup on Colab; the timing command includes warm-up, forward/backward, and Canny convergence overhead. No full training is started.


In [ ]:
!python -m unittest discover -s tests -v
!python tests/benchmark_canny_v1.py --device cuda --size 256 --batch-size 8 --steps 10
